# Experiment 9d - Chronos-2 covariate FINE-TUNE (native `fit()`)

**Method switch from the AutoGluon attempt:** AutoGluon proved env-incompatible with
Kaggle's numpy-2.x base (needs numpy<2 -> breaks pandas/scipy ABI; 3 env breakages).
But `chronos-forecasting` 2.2 has an **undocumented native `Chronos2Pipeline.fit()`** -
full or LoRA fine-tuning via an internal HF Trainer. This runs on the **same stack that
worked cleanly in 9c** (sine test passed), so no new heavy installs.

**Plan (honest, low-overfit):**
- Fine-tune **with the 11 known covariates** (the multivariate signal; the only Chronos
  config that competed here, LB 14.93).
- **Per-fold rolling-origin:** fine-tune only on history before each fold's test window;
  the held-out 336h test window is never trained on -> honest WAPE.
- `finetune_mode="full"`, modest `NUM_STEPS` + low LR to avoid catastrophic forgetting.
- Predict with the fine-tuned pipeline via `predict_df` (the proven 9c covariate path).

**Reality check (stated up front):** the LB ceiling for fine-tuned-covariate Chronos here
is ~14.93 - *worse* than your shipped 14.68 ensemble / 14.75 MLP. So solo it won't win;
its only value is as a **decorrelated stacking arm** (gate: residual r vs MLP < 0.85).

**Risk:** `fit()`'s input format is undocumented. §4 is a **5-step smoke test** to validate
the covariate input format cheaply before the real per-fold runs - if it errors, we fix the
input dict in one shot (don't run §5+ until §4 passes).


---
## §0. Configuration

In [1]:
from pathlib import Path

DATA_DIR   = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR = Path("/kaggle/working")
# Local:
# DATA_DIR   = Path("data")
# OUTPUT_DIR = Path("experiment-9")

MLP_OOF_PATH  = Path("/kaggle/input/datasets/xitij27/oof-data/oof_predictions_7e_mlp_gated.csv")
LGBM_OOF_PATH = Path("/kaggle/input/datasets/xitij27/oof-data/oof_predictions_9a_spike.csv")
# Local: experiment-7/oof_predictions_7e_mlp_gated.csv  /  experiment-9/oof_predictions_9a_spike.csv

TRAIN_PATH = DATA_DIR / "train.csv"

N_FOLDS         = 3
EVAL_HORIZON    = 336
CHRONOS_MODEL   = "amazon/chronos-2"
QUANTILE_LEVELS = [0.1, 0.5, 0.9]

DOMAIN_FEAT_COLS = [
    "workload_intensity", "demand_forecast", "upstream_quality_forecast",
    "promotion_intensity", "shock_risk", "maintenance_known",
    "unit_reliability_forecast", "queue_pressure_forecast",
    "network_pressure_forecast", "event_load_forecast",
    "service_irregularity_risk_forecast",
]
KNOWN_COV = DOMAIN_FEAT_COLS

# Fine-tune knobs (start modest; raise NUM_STEPS once the smoke test passes)
FINETUNE_MODE = "full"     # "full" (no extra deps) or "lora" (needs peft)
NUM_STEPS     = 2000
LEARNING_RATE = 1e-5       # full-finetune; keep low to avoid catastrophic forgetting
BATCH_SIZE    = 8         # 64 OOMed on one T4 (each task = target + 11 covariate series)
CONTEXT_LENGTH = 1024     # None=model default is long -> OOM; this is the biggest memory lever
USE_BOTH_GPUS  = True     # Kaggle gives 2x T4; Trainer DataParallel splits the batch across them
# If it still OOMs: drop CONTEXT_LENGTH to 512 and/or BATCH_SIZE to 4, or set FINETUNE_MODE="lora".

# Decision thresholds - pre-registered
BEATS_GATED       = 14.59
BEATS_ENSEMBLE    = 14.47
BLEND_R_THRESHOLD = 0.85
CHRONOS_LB_REF    = 14.93

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"fine-tune: mode={FINETUNE_MODE}, {NUM_STEPS} steps, lr={LEARNING_RATE}, batch={BATCH_SIZE}, covariates={len(KNOWN_COV)}")


fine-tune: mode=full, 2000 steps, lr=1e-05, batch=8, covariates=11


---
## §1. Setup (same working stack as 9c - NO AutoGluon)

In [2]:
import subprocess, sys
# torchvision/torchaudio get version-mismatched when other installs move torch, and
# transformers eagerly imports torchvision ("operator torchvision::nms does not exist").
# chronos needs neither - drop them so the transformers import is clean.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"], check=False)
# The exact 9c install that ran cleanly on Kaggle's numpy-2.x base.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "chronos-forecasting>=2.0", "jaxtyping"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.41,<5", "einops>=0.7.0", "accelerate>=0.34,<2", "peft"], check=False)

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # reduce fragmentation
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from chronos import Chronos2Pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "|", (torch.cuda.get_device_name(0) if DEVICE=="cuda" else "CPU"))
print("GPU kernel sanity:", float((torch.randn(8, device="cuda")+1).sum()) if DEVICE=="cuda" else "cpu")

def wape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps) * 100


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.26 requires torchvision, which is not installed.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.

torch 2.10.0+cu128 | Tesla T4
GPU kernel sanity: 6.306571960449219


---
## §2. Load + impute + long frame

In [3]:
train_df = pd.read_csv(TRAIN_PATH, parse_dates=["timestamp"])
def impute(df, cols):
    df = df.sort_values(["series_id","timestamp"]).copy()
    df[cols] = df.groupby("series_id", group_keys=False)[cols].transform(lambda x: x.ffill().bfill())
    return df
base = impute(train_df, DOMAIN_FEAT_COLS).sort_values(["series_id","timestamp"]).reset_index(drop=True)
base["pos"] = base.groupby("series_id").cumcount()
T_PER_SERIES = int(base.groupby("series_id").size().iloc[0])
print(f"base {base.shape}, len/series {T_PER_SERIES}")
def fold_origin(k): return T_PER_SERIES - (N_FOLDS - k) * EVAL_HORIZON
CTX_COLS = ["series_id","timestamp","target"] + KNOWN_COV
FUT_COLS = ["series_id","timestamp"] + KNOWN_COV


base (414720, 26), len/series 4320


---
## §3. Build `fit()` inputs + load reference OOFs

`fit()` takes a list of per-series dicts. Known covariates are provided as full-length arrays under `past_covariates` (the windowing sampler slices context+future from them). If the smoke test (§4) rejects this format, the error tells us the exact fix.

In [4]:
def build_inputs(df):
    """List of per-series dicts for Chronos2Pipeline.fit().
    target: full-length array; covariates: full-length arrays (known throughout)."""
    out = []
    for _, g in df.groupby("series_id", sort=False):
        g = g.sort_values("timestamp")
        out.append({
            "target": g["target"].to_numpy(dtype="float32"),
            "past_covariates": {c: g[c].to_numpy(dtype="float32") for c in KNOWN_COV},
        })
    return out

ref = None
if MLP_OOF_PATH.exists():
    ref = pd.read_csv(MLP_OOF_PATH, parse_dates=["timestamp"])[["series_id","timestamp","fold","target","pred_mlp_gated"]]
if LGBM_OOF_PATH.exists():
    lg = pd.read_csv(LGBM_OOF_PATH, parse_dates=["timestamp"])[["series_id","timestamp","fold","pred_mae"]]
    ref = lg if ref is None else ref.merge(lg, on=["series_id","timestamp","fold"], how="left")
print("reference OOF:", None if ref is None else ref.shape)


reference OOF: (96768, 6)


---
## §4. SMOKE TEST - validate `fit()` input format (5 steps, 4 series)

Cheap guard before the real runs. If this raises on the covariate format, paste the error and we fix `build_inputs` once. Do NOT run §5+ until this prints OK.

In [5]:
pipe = Chronos2Pipeline.from_pretrained(CHRONOS_MODEL, device_map=DEVICE)

smoke_sids = sorted(base["series_id"].unique())[:4]
smoke_df = base[(base["series_id"].isin(smoke_sids)) & (base["pos"] < fold_origin(0))]
smoke_inputs = build_inputs(smoke_df[CTX_COLS])
print(f"smoke: {len(smoke_inputs)} series, target len {len(smoke_inputs[0]['target'])}, "
      f"covariates {list(smoke_inputs[0]['past_covariates'])[:3]}...")

ft_smoke = pipe.fit(
    inputs=smoke_inputs, prediction_length=EVAL_HORIZON,
    finetune_mode=FINETUNE_MODE, num_steps=5, batch_size=4,
    learning_rate=LEARNING_RATE, output_dir=str(OUTPUT_DIR/"ag_smoke"),
)
# quick predict to confirm the fine-tuned pipeline works end-to-end
sm_origin = fold_origin(0)
sm_ctx = base[(base.series_id.isin(smoke_sids)) & (base.pos < sm_origin)][CTX_COLS]
sm_fut = base[(base.series_id.isin(smoke_sids)) & (base.pos >= sm_origin) & (base.pos < sm_origin+EVAL_HORIZON)][FUT_COLS]
sm_pred = ft_smoke.predict_df(sm_ctx, future_df=sm_fut, prediction_length=EVAL_HORIZON,
                              quantile_levels=QUANTILE_LEVELS, id_column="series_id",
                              timestamp_column="timestamp", target="target")
print("OK smoke fit+predict ran. pred cols:", list(sm_pred.columns))
print(">>> If this printed OK, run §5. If it errored on covariates, paste the error.")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

smoke: 4 series, target len 3312, covariates ['workload_intensity', 'demand_forecast', 'upstream_quality_forecast']...


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss


OK smoke fit+predict ran. pred cols: ['series_id', 'timestamp', 'target_name', 'predictions', '0.1', '0.5', '0.9']
>>> If this printed OK, run §5. If it errored on covariates, paste the error.


---
## §5. Per-fold covariate fine-tune + forecast

In [6]:
oof_rows, per_fold = [], []
for k in range(N_FOLDS):
    origin = fold_origin(k)
    train_slice = base[base["pos"] < origin][CTX_COLS]
    futr = base[(base["pos"] >= origin) & (base["pos"] < origin + EVAL_HORIZON)]
    print(f"\n===== Fold {k}: fine-tune on <{origin} ({train_slice['series_id'].nunique()} series), test next {EVAL_HORIZON} =====")

    train_inputs = build_inputs(train_slice)
    ft = pipe.fit(
        inputs=train_inputs, prediction_length=EVAL_HORIZON,
        finetune_mode=FINETUNE_MODE, num_steps=NUM_STEPS, batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE, context_length=CONTEXT_LENGTH,
        disable_data_parallel=not USE_BOTH_GPUS,
        output_dir=str(OUTPUT_DIR / f"ft_fold{k}"),
    )

    ctx = train_slice
    fut = futr[FUT_COLS]
    pred = ft.predict_df(ctx, future_df=fut, prediction_length=EVAL_HORIZON,
                         quantile_levels=QUANTILE_LEVELS, id_column="series_id",
                         timestamp_column="timestamp", target="target")
    mc = "0.5" if "0.5" in pred.columns else ("predictions" if "predictions" in pred.columns else pred.columns[-1])
    hi = "0.9" if "0.9" in pred.columns else mc
    pred = pred.rename(columns={mc: "pred_chronos_ft", hi: "pred_chronos_ft_q90"})
    pred["pred_chronos_ft"] = pred["pred_chronos_ft"].clip(lower=0)

    merged = futr[["series_id","timestamp","target"]].merge(
        pred[["series_id","timestamp","pred_chronos_ft","pred_chronos_ft_q90"]],
        on=["series_id","timestamp"], how="left")
    merged["lead_h"] = merged.groupby("series_id").cumcount() + 1
    merged["fold"] = k
    per_fold.append(wape(merged["target"], merged["pred_chronos_ft"]))
    oof_rows.append(merged)
    print(f"  fold {k}: fine-tuned Chronos-2 WAPE = {per_fold[-1]:.3f}")

oof_ft = pd.concat(oof_rows, ignore_index=True)
FT_WAPE = float(np.mean(per_fold))
print(f"\nFine-tuned Chronos-2 CV-WAPE = {FT_WAPE:.3f}  (zero-shot ~35, gated-MLP {BEATS_GATED}, LB ref {CHRONOS_LB_REF})")



===== Fold 0: fine-tune on <3312 (96 series), test next 336 =====


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
100,0.584300
200,0.543300
300,0.535900
400,0.550800
500,0.578700
600,0.543300
700,0.546900
800,0.550200
900,0.536600
1000,0.504600


  fold 0: fine-tuned Chronos-2 WAPE = 17.224

===== Fold 1: fine-tune on <3648 (96 series), test next 336 =====


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
100,0.583000
200,0.536600
300,0.540300
400,0.552900
500,0.570500
600,0.554000
700,0.527900
800,0.550400
900,0.555700
1000,0.512200


  fold 1: fine-tuned Chronos-2 WAPE = 20.723

===== Fold 2: fine-tune on <3984 (96 series), test next 336 =====


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
100,0.591200
200,0.553500
300,0.560600
400,0.534600
500,0.549700
600,0.582000
700,0.522400
800,0.552400
900,0.540700
1000,0.552400


  fold 2: fine-tuned Chronos-2 WAPE = 18.174

Fine-tuned Chronos-2 CV-WAPE = 18.707  (zero-shot ~35, gated-MLP 14.59, LB ref 14.93)


---
## §6. Evaluation - metrics, residual r, per-horizon

In [7]:
def all_metrics(y, p):
    err = np.asarray(y,float) - np.asarray(p,float)
    return dict(MAE=float(np.mean(np.abs(err))), MSE=float(np.mean(err**2)),
                RMSE=float(np.sqrt(np.mean(err**2))), WAPE=float(100*np.sum(np.abs(err))/np.sum(np.abs(y))))

J = oof_ft.copy()
if ref is not None:
    J = J.merge(ref.drop(columns=[c for c in ["target"] if c in ref.columns]),
                on=["series_id","timestamp","fold"], how="left")
y = J["target"].to_numpy()
panel = [{"model":"Chronos-2 FT", **all_metrics(y, J["pred_chronos_ft"])}]
if "pred_mlp_gated" in J: panel.append({"model":"gated-MLP", **all_metrics(y, J["pred_mlp_gated"])})
if "pred_mae" in J:       panel.append({"model":"LGBM-MAE",  **all_metrics(y, J["pred_mae"])})
print(pd.DataFrame(panel).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

def rr(c): return J["target"] - J[c]
for col, name in [("pred_mlp_gated","gatedMLP"), ("pred_mae","LGBM")]:
    if col in J:
        r = float(np.corrcoef(rr(col), rr("pred_chronos_ft"))[0,1])
        print(f"r(Chronos-FT, {name}) = {r:.4f}  ({'DECORRELATED' if r < BLEND_R_THRESHOLD else 'correlated'})")

print("\nPer-horizon WAPE:")
for lo,hi in [(1,1),(2,6),(7,24),(25,72),(73,168),(169,336)]:
    sl = J[(J.lead_h>=lo)&(J.lead_h<=hi)]
    ex = f"  mlp={wape(sl['target'],sl['pred_mlp_gated']):.2f}" if "pred_mlp_gated" in J else ""
    print(f"  {lo:>3}..{hi:<3}h (n={len(sl):>5}): ft={wape(sl['target'],sl['pred_chronos_ft']):.2f}{ex}")


       model    MAE     MSE   RMSE    WAPE
Chronos-2 FT 1.9612 10.3875 3.2230 18.5820
   gated-MLP 1.5349  8.7478 2.9577 14.5424
    LGBM-MAE 1.5499  8.8537 2.9755 14.6848
r(Chronos-FT, gatedMLP) = 0.8995  (correlated)
r(Chronos-FT, LGBM) = 0.9069  (correlated)

Per-horizon WAPE:
    1..1  h (n=  288): ft=19.12  mlp=25.69
    2..6  h (n= 1440): ft=27.33  mlp=21.48
    7..24 h (n= 5184): ft=16.42  mlp=12.26
   25..72 h (n=13824): ft=21.57  mlp=16.77
   73..168h (n=27648): ft=17.62  mlp=13.85
  169..336h (n=48384): ft=18.54  mlp=14.55


---
## §7. Stacking blend + §8 save/verdict

In [8]:
if "pred_mlp_gated" in J:
    pa = J["pred_mlp_gated"].to_numpy(); pb = J["pred_chronos_ft"].to_numpy()
    def cvw(w):
        J["_b"]=w*pa+(1-w)*pb; v=np.mean([wape(g["target"],g["_b"]) for _,g in J.groupby("fold")]); J.drop(columns="_b",inplace=True); return float(v)
    grid=np.round(np.arange(0,1.0001,0.025),3)
    bw,bv=min(((w,cvw(w)) for w in grid), key=lambda t:t[1])
    print(f"gated-MLP solo {cvw(1.0):.3f} | Chronos-FT solo {cvw(0.0):.3f} | best blend {bv:.3f} at w_mlp={bw:.3f} (Δ vs 14.47: {BEATS_ENSEMBLE-bv:+.3f})")

oof_ft[["series_id","timestamp","target","pred_chronos_ft","pred_chronos_ft_q90","lead_h","fold"]].to_csv(
    OUTPUT_DIR/"oof_predictions_9d_chronos_ft.csv", index=False)
print("OOF saved.")

r_mlp = float(np.corrcoef(rr("pred_mlp_gated"), rr("pred_chronos_ft"))[0,1]) if "pred_mlp_gated" in J else float("nan")
print("\n=== Verdict ===")
print(f"  Chronos-2 FT CV-WAPE {FT_WAPE:.3f} | MSE {all_metrics(y,J['pred_chronos_ft'])['MSE']:.3f} | r vs MLP {r_mlp:.3f}")
if FT_WAPE <= BEATS_GATED: print("  BEATS gated-MLP solo - refit on full train for a submission.")
if r_mlp == r_mlp and r_mlp < BLEND_R_THRESHOLD: print(f"  DECORRELATED (r<{BLEND_R_THRESHOLD}) - ship the §7 blend if it beats 14.47.")
if (FT_WAPE > BEATS_GATED) and not (r_mlp < BLEND_R_THRESHOLD):
    print("  Neither beats MLP nor decorrelates -> raise NUM_STEPS once; else document & close Frontier 2.")


gated-MLP solo 14.593 | Chronos-FT solo 18.707 | best blend 14.588 at w_mlp=0.975 (Δ vs 14.47: -0.118)
OOF saved.

=== Verdict ===
  Chronos-2 FT CV-WAPE 18.707 | MSE 10.388 | r vs MLP 0.899
  Neither beats MLP nor decorrelates -> raise NUM_STEPS once; else document & close Frontier 2.
